# Task 4 & 5: Portfolio Optimization and Strategy Backtesting

This notebook implements Modern Portfolio Theory (MPT) to optimize a portfolio of **TSLA**, **BND**, and **SPY** by combining forecasted returns with historical data. Then, we backtest this optimized strategy against a 60/40 benchmark.

In [ ]:
import sys
sys.path.append('../')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_local_data
from src.eda_utils import calculate_daily_returns
from src.portfolio import (
    optimize_portfolio, generate_efficient_frontier, get_portfolio_metrics
)
from src.backtest import run_backtest, calculate_backtest_metrics

## 1. Load Data and Prepare Returns
We use the forecasted return of TSLA and historical returns of BND and SPY.

In [ ]:
tickers = ["TSLA", "BND", "SPY"]
data_dict = load_local_data(tickers, input_dir="../data/processed")

# Calculate daily historical returns
returns_df = pd.DataFrame({
    t: calculate_daily_returns(data_dict[t]) for t in tickers
}).dropna()

# Expected returns: For TSLA, load the forecasted returns from task 3.
# Here we calculate the average forecasted daily return and annualize it.
future_forecast = pd.read_csv("../data/processed/future_forecast.csv", index_col=0)
# Let's use the ARIMA future forecast mean
tsla_forecast_returns = future_forecast["ARIMA_Mean"].pct_change().dropna()
tsla_expected_return = tsla_forecast_returns.mean() * 252

expected_returns = pd.Series({
    "TSLA": tsla_expected_return,
    "BND": returns_df["BND"].mean() * 252,
    "SPY": returns_df["SPY"].mean() * 252
})
print("Expected Annualized Returns:")
print(expected_returns)

# Covariance Matrix (annualized)
cov_matrix = returns_df.cov() * 252
print("\nCovariance Matrix (Annualized):")
print(cov_matrix)

## 2. Correlation and Covariance Heatmap

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(returns_df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Heatmap", fontsize=12)
plt.show()

## 3. Generate Efficient Frontier
We run simulations and solve for the optimal portfolios.

In [ ]:
weights_max_sharpe, weights_min_vol = optimize_portfolio(expected_returns, cov_matrix)
print("Max Sharpe Portfolio Weights:", weights_max_sharpe)
print("Min Volatility Portfolio Weights:", weights_min_vol)

# Run simulations for plotting the frontier
results, weights_record = generate_efficient_frontier(expected_returns, cov_matrix, num_portfolios=5000)

# Retrieve optimized portfolios metrics
w_ms = [weights_max_sharpe[t] for t in tickers]
ret_ms, vol_ms, sharpe_ms = get_portfolio_metrics(w_ms, expected_returns, cov_matrix)

w_mv = [weights_min_vol[t] for t in tickers]
ret_mv, vol_mv, sharpe_mv = get_portfolio_metrics(w_mv, expected_returns, cov_matrix)

plt.figure(figsize=(10, 6))
sc = plt.scatter(results[1, :], results[0, :], c=results[2, :], cmap='viridis', s=10, alpha=0.3)
plt.colorbar(sc, label='Sharpe Ratio')
plt.scatter(vol_ms, ret_ms, color='red', marker='*', s=200, label='Max Sharpe Ratio')
plt.scatter(vol_mv, ret_mv, color='blue', marker='X', s=200, label='Min Volatility')
plt.title("Efficient Frontier", fontsize=14)
plt.xlabel("Annualized Volatility (Risk)", fontsize=12)
plt.ylabel("Expected Return", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

## 4. Task 5: Backtesting
We test our Max Sharpe Ratio strategy against a static benchmark of 60% SPY / 40% BND during the 2025 period (rebalanced monthly).

In [ ]:
backtest_returns = returns_df.loc["2025-01-01":"2026-01-01"]

# Optimized Strategy (Max Sharpe, Monthly Rebalanced)
strat_value, _ = run_backtest(backtest_returns, weights_max_sharpe, rebalance_monthly=True)
strat_metrics = calculate_backtest_metrics(strat_value)

# Benchmark (60% SPY / 40% BND, Buy & Hold)
benchmark_weights = {"TSLA": 0.0, "SPY": 0.6, "BND": 0.4}
bench_value, _ = run_backtest(backtest_returns, benchmark_weights, rebalance_monthly=False)
bench_metrics = calculate_backtest_metrics(bench_value)

metrics_compare = pd.DataFrame({
    "Optimized Strategy": strat_metrics,
    "60/40 Benchmark": bench_metrics
})
print(metrics_compare)

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot((strat_value / strat_value.iloc[0]) - 1, label="Optimized Strategy", color='blue')
plt.plot((bench_value / bench_value.iloc[0]) - 1, label="Benchmark (60/40 SPY/BND)", color='grey', linestyle='--')
plt.title("Backtesting Performance: Cumulative Returns Comparison (2025)", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Cumulative Return", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()